In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics
print("\n✅ Cài xong.")


## Đã có sẵn `best.pt`? Bỏ ảnh + weights vào đâu

Trên Kaggle, mọi thứ bạn "Add Data" đều được mount **read-only** vào `/kaggle/input/<tên-dataset>/...`.
Không có chỗ nào để kéo-thả ảnh trực tiếp vào notebook — bạn phải tạo dataset rồi gắn vào notebook:

1. Bấm **"+ Add Input"** (hoặc "Add Data") ở panel bên phải notebook.
2. Tab **Upload** → kéo thả file `best.pt` vào → đặt tên dataset, ví dụ `plate-yolo-weights` → Create.
3. Làm tương tự với **ảnh test** (upload 1 hoặc nhiều ảnh, hoặc cả 1 thư mục ảnh) → đặt tên dataset,
   ví dụ `plate-test-images`.
4. Sau khi 2 dataset trên hiện trong panel "Input" bên phải, chạy cell đầu tiên của notebook này —
   nó in ra toàn bộ đường dẫn file trong `/kaggle/input`, copy đường dẫn `best.pt` và ảnh từ đó.

Cell dưới **tự tìm** `best.pt` và ảnh trong `/kaggle/input` giúp bạn — nếu tìm thấy nhiều hơn 1 hoặc
không tìm thấy, tự gán tay biến tương ứng.

In [ ]:
import glob

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

# ⭐ Tự tìm best.pt trong /kaggle/input. Nếu có nhiều hơn 1 hoặc muốn chỉ định thủ công,
#   gán thẳng: BEST_PLATE_WEIGHTS = '/kaggle/input/plate-yolo-weights/best.pt'
_found_weights = glob.glob('/kaggle/input/**/*.pt', recursive=True)

BEST_PLATE_WEIGHTS = None
if len(_found_weights) == 1:
    BEST_PLATE_WEIGHTS = _found_weights[0]
    print("✅ Tìm thấy weights:", BEST_PLATE_WEIGHTS)
elif len(_found_weights) > 1:
    print("⚠️ Tìm thấy nhiều file .pt, hãy gán tay BEST_PLATE_WEIGHTS cho đúng cái cần dùng:")
    for p in _found_weights:
        print("  -", p)
else:
    print("ℹ️ Chưa thấy file .pt nào trong /kaggle/input — nếu chưa upload best.pt thì bỏ qua, notebook sẽ train mới ở dưới.")

# ⭐ Tự tìm ảnh test (loại trừ ảnh nằm trong thư mục train/valid của dataset, nếu có).
_found_imgs = [
    p for p in glob.glob('/kaggle/input/**/*', recursive=True)
    if p.lower().endswith(IMG_EXTS) and '/train/' not in p and '/valid/' not in p and '/test/' not in p
]
print(f"\nℹ️ Tìm thấy {len(_found_imgs)} ảnh test trong /kaggle/input (không tính ảnh trong thư mục train/valid/test của dataset).")

# ⭐ Muốn chỉ định thủ công danh sách/đường dẫn ảnh test thì gán tay TEST_IMAGES, ví dụ:
#   TEST_IMAGES = ['/kaggle/input/plate-test-images/xe1.jpg']
TEST_IMAGES = _found_imgs


## Dataset để train (chỉ cần nếu **chưa có** `best.pt` sẵn)

Nếu ở trên đã tìm thấy `best.pt`, bạn có thể **bỏ qua cả phần train này**, chạy thẳng xuống phần
"Test detect" ở cuối notebook.

Nếu chưa có, cần một dataset detect biển số ở định dạng YOLOv8 (thư mục `train/images`,
`train/labels`... kèm file `data.yaml`), upload qua "Add Data" giống như trên.

In [ ]:
# ⭐ Tự tìm data.yaml trong /kaggle/input. Nếu có nhiều hơn 1 hoặc muốn chỉ định thủ công,
#   gán thẳng: DATA_YAML = '/kaggle/input/.../data.yaml'
_found_yaml = glob.glob('/kaggle/input/**/data.yaml', recursive=True)

DATA_YAML = None
if len(_found_yaml) == 1:
    DATA_YAML = _found_yaml[0]
    print("✅ Tìm thấy data.yaml:", DATA_YAML)
elif len(_found_yaml) > 1:
    print("⚠️ Tìm thấy nhiều data.yaml, hãy gán tay DATA_YAML cho đúng cái cần dùng:")
    for p in _found_yaml:
        print("  -", p)
elif BEST_PLATE_WEIGHTS:
    print("ℹ️ Không có data.yaml, nhưng đã có best.pt nên không cần train — bỏ qua các cell dataset/train bên dưới.")
else:
    print("⚠️ Không tìm thấy data.yaml nào, và cũng chưa có best.pt — hãy gán tay DATA_YAML để train mới.")


### Xem thử vài ảnh + nhãn trước khi train

Kiểm tra nhanh vài ảnh cùng bounding box đã gán nhãn, để chắc dataset đúng (khung bao quanh đúng
biển số) trước khi tốn thời gian train. Tự bỏ qua nếu không có `data.yaml`.

In [ ]:
import cv2
import matplotlib.pyplot as plt

def parse_label_line(parts, w, h):
    """Trả về (cls, x1, y1, x2, y2) dù dòng là bbox (5 số) hay polygon (>5 số)."""
    cls = int(float(parts[0]))
    vals = list(map(float, parts[1:]))
    if len(vals) == 4:
        xc, yc, bw, bh = vals
        x1 = (xc - bw / 2) * w; y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w; y2 = (yc + bh / 2) * h
    else:
        xs = [vals[i] * w for i in range(0, len(vals), 2)]
        ys = [vals[i] * h for i in range(1, len(vals), 2)]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
    return cls, int(x1), int(y1), int(x2), int(y2)

if DATA_YAML:
    import yaml
    with open(DATA_YAML) as f:
        data_cfg = yaml.safe_load(f)
    print("Số lớp:", data_cfg.get('nc'))
    print("Danh sách lớp:", data_cfg.get('names'))
    names = data_cfg.get('names')

    train_img_dir = os.path.join(os.path.dirname(DATA_YAML), 'train', 'images')
    train_lbl_dir = os.path.join(os.path.dirname(DATA_YAML), 'train', 'labels')
    sample_imgs = sorted(glob.glob(os.path.join(train_img_dir, '*')))[:6]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, img_path in zip(axes.flat, sample_imgs):
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl_path = os.path.join(train_lbl_dir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.split()
                    if not parts:
                        continue
                    cls, x1, y1, x2, y2 = parse_label_line(parts, w, h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    label = names[cls] if cls < len(names) else str(cls)
                    cv2.putText(img, label, (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("⏭️ Không có data.yaml — bỏ qua preview dataset.")


### Train YOLOv8 detect vị trí biển số

Chỉ detect **khung bao (bounding box)** của biển số trong ảnh, không đọc ký tự. Tự bỏ qua nếu ở
trên đã tìm thấy `best.pt` (biến `BEST_PLATE_WEIGHTS`).

In [ ]:
from ultralytics import YOLO

if BEST_PLATE_WEIGHTS and os.path.exists(BEST_PLATE_WEIGHTS):
    print("✅ Dùng model có sẵn, bỏ qua train:", BEST_PLATE_WEIGHTS)
else:
    assert DATA_YAML, "Chưa có best.pt lẫn data.yaml — cần 1 trong 2 để có model. Xem lại 2 cell config ở đầu notebook."

    plate_model = YOLO('yolov8n.pt')

    plate_model.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,           # ⭐ ảnh cả xe/khung hình lớn hơn nhiều so với detect ký tự -> cần imgsz lớn
        batch=16,
        patience=20,         # ⭐ early stop nếu không cải thiện sau 20 epoch
        project='/kaggle/working/yolo_plate_runs',
        name='train',
    )

    # ⭐ train() trả về kiểu khác nhau tuỳ version ultralytics (có bản trả dict,
    #   không có .save_dir) -> lấy qua plate_model.trainer.save_dir cho ổn định
    BEST_PLATE_WEIGHTS = str(plate_model.trainer.save_dir / 'weights' / 'best.pt')
    print("✅ Train xong, weights tốt nhất tại:", BEST_PLATE_WEIGHTS)


## Test detect + crop biển số ra khỏi ảnh gốc

Chạy model lên các ảnh test tìm được ở `TEST_IMAGES` (cell config đầu notebook), vẽ khung phát
hiện được, và crop riêng vùng biển số ra (dùng để đưa tiếp sang bước đọc ký tự sau này, ví dụ
notebook `yolo_character_recognition.ipynb`).

In [ ]:
assert BEST_PLATE_WEIGHTS, "Chưa có BEST_PLATE_WEIGHTS — train ở cell trên trước, hoặc gán tay đường dẫn best.pt."
assert TEST_IMAGES, "Chưa có ảnh test nào — upload ảnh qua Add Data rồi gán tay biến TEST_IMAGES."

plate_model_trained = YOLO(BEST_PLATE_WEIGHTS)

def crop_plates(result, conf_thres=0.4):
    """Trả về danh sách ảnh (numpy array, BGR) đã crop từng biển số phát hiện được trong 1 result."""
    crops = []
    img = result.orig_img
    boxes = result.boxes
    if boxes is None:
        return crops
    for b in boxes:
        conf = float(b.conf[0])
        if conf < conf_thres:
            continue
        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
        crops.append(img[y1:y2, x1:x2])
    return crops

for TEST_IMG in TEST_IMAGES:
    result = plate_model_trained(TEST_IMG, conf=0.4, verbose=False)[0]

    annotated = result.plot()
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(TEST_IMG))
    plt.axis('off')
    plt.show()

    plate_crops = crop_plates(result)
    print(f"📋 {os.path.basename(TEST_IMG)}: phát hiện {len(plate_crops)} biển số")
    for crop in plate_crops:
        plt.figure(figsize=(6, 2))
        plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()
